# DIY Copilot
## Lesson: Context Engineering

Everything a model knows about your task is in the **context window** of a single request. The model is stateless: with every request the harness sends the complete context again – the system prompt, the tool declarations, the conversation so far, every tool result:

```
┌──────────────────────────────────────── one request ─────────────────────────────────────────┐
│ system prompt │ tool declarations │ instructions, skills, code │ conversation: user, assistant, │
│               │                   │                            │ tool calls, tool results, ...  │
└──────────────────────────────────────────────────────────────────────────────────────────────┘
  stable: the same in every request ──────────────────────────►   grows with every step ───────►
```

**Context engineering** is the design of this context: *what* goes in, *when*, in *which order* – and what stays out. Prompt engineering is about the wording; context engineering treats the context window as a limited resource of the harness. More context is not better: every token costs money and latency, and with long, noisy contexts the answers get worse (*context rot*). That's why coding assistants such as Claude Code, GitHub Copilot or Cursor put a lot of effort into it.

During this lesson you will learn:
* where the tokens of a coding session go (and why the costs grow faster than the conversation)
* how **prompt caching** makes the stable part cheaper – and how a harness can break it
* how **progressive disclosure** keeps the stable part small: the assistant gets an index of **skills** and loads their instructions only when it needs them
* how **history condensing** keeps the growing part small

### Set up the environment

In [ ]:
# >>> SETUP <<<
# In Google Colab, this cell
#   1. clones the workshop repository,
#   2. installs its dependencies into the Python environment of this notebook (with uv),
#   3. makes the shared code (lib) importable.
# Locally (after `uv sync` in the repository), only step 3 is needed.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/openknowledge/diy-copilot-bootcamp"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/diy-copilot")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, str(REPO_DIR / "setup" / "install_dependencies.py")], check=True)
else:
    # The repository root is the first parent directory containing the shared code
    REPO_DIR = next(p for p in Path.cwd().resolve().parents if (p / "lib").is_dir() and (p / "pyproject.toml").exists())

# The repository is not an installed package, so we add it to the module search path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Workshop repository: {REPO_DIR}")

In [ ]:
import tempfile
from datetime import datetime
from pathlib import Path

import litellm
from litellm import completion

from lib.constants import CODEBASE_PATH, SKILLS_PATH
from lib.context_engineering import (
    READ_FILE_TOOL,
    build_work_412_session,
    cached_tokens,
    clear_old_tool_results,
    count_tokens,
    format_transcript,
    history_tokens,
    load_skills,
    print_conversation,
    print_usage,
    read_file,
    run_assistant,
    run_tool_loop,
    token_breakdown,
    token_growth,
    turn_starts,
)
from lib.environment import load_environment
from lib.retry import call_with_retry

default_models = load_environment()
generation_model = default_models["generation_model"]
print(f"Generation model: {generation_model}")

### The coding session we use in this lesson

A developer of Team Workplace fixes the bug `WORK-412` in Roomie (`500 Internal Server Error` when a booking is created without timezone) with the help of the assistant. The assistant has a tool `read_file`, like the tools in the function calling lesson.

We don't run this session live – that would cost a lot of requests. Instead, we load the conversation as a list of messages, exactly as the harness would have collected it: the developer pastes a server log, they make a decision together, the assistant reads three files, the developer pastes a test output. Read through the messages to know the story.

In [ ]:
session = build_work_412_session()
print_conversation(session)

### Example: Where do the tokens go?

Let's measure the session. First: which parts of the context use the most tokens?

>**Note:** `count_tokens` estimates the tokens locally (for Gemini about 20% too low). That's good enough to compare. The exact numbers are reported by the provider in `response.usage`.

In [ ]:
token_breakdown(session, [READ_FILE_TOOL])

The developer's questions and the assistant's answers are tiny. Almost all tokens are **material**: pasted logs, test output, file contents.

Second: the harness sends a request after every developer message *and* after every tool result – each time with everything that came before. How many tokens does it send in total?

In [ ]:
token_growth(session, [READ_FILE_TOOL])

What did we learn?
* Every request resends everything before it. The tokens sent in total grow **quadratically** with the length of the session – the server log from the first message is paid for in every single request.
* There are three levers, and each one is an exercise of this lesson:
  1. Make the **stable part** cheaper → *prompt caching* (exercise 01)
  2. Make the **stable part** smaller → *progressive disclosure* (exercise 02)
  3. Keep the **growing part** small → *history condensing* (exercise 03)

### Exercise 01: Prompt caching

To process a prompt, the model computes internal states for every token. If many requests start with exactly the same tokens, the provider can store these states and reuse them: **prompt caching**. Cached tokens are much cheaper (with Gemini: 10% of the normal price) and the request can be faster.

Gemini caches *implicitly*: you don't have to do anything, it only reports in `usage.prompt_tokens_details.cached_tokens` how many tokens came from the cache. Other providers work similarly (OpenAI caches automatically, Anthropic needs explicit cache markers in the request). The rules are always the same:

* Only an identical **prefix** is cached. A single changed token at the beginning invalidates everything after it.
* The prefix must have a minimum length (a few thousand tokens), and the cache expires after a few minutes without use.
* The first request with a new prefix is always a miss – it fills the cache. Caching is best effort, not guaranteed.

Coding assistants make heavy use of it: the system prompt, the tool declarations and the project instructions are the same in every request of a session. Let's put the whole Roomie code base into the system prompt (about 12,000 tokens) and ask two different questions.

In [ ]:
code_base = "\n\n".join(
    f"# File: {path.relative_to(CODEBASE_PATH)}\n{path.read_text()}"
    for path in sorted(CODEBASE_PATH.rglob("*.py"))
    if "__pycache__" not in path.parts
)
ASSISTANT_INSTRUCTIONS = (
    "You are a coding assistant for Team Workplace, the team that owns Roomie, the meeting room booking service. "
    "Answer questions about the code below. Keep your answers short."
)
SYSTEM_PROMPT_WITH_CODE = f"{ASSISTANT_INSTRUCTIONS}\n\n## Code base\n\n{code_base}"
print(f"System prompt: about {count_tokens(SYSTEM_PROMPT_WITH_CODE)} tokens")


def ask(messages: list[dict]):
    return call_with_retry(completion, model=generation_model, messages=messages, temperature=0.0, max_tokens=100)


for question in ["Which function checks the business hours?", "Where is the API key of a request checked?"]:
    response = ask([{"role": "system", "content": SYSTEM_PROMPT_WITH_CODE}, {"role": "user", "content": question}])
    print_usage(question[:28], response)

The second request should report about 10,000 cached tokens. (If the first one does too, somebody – maybe you – used the same prefix a few minutes ago.)

Many harnesses add **dynamic information** to the system prompt: the current date, the name of the developer, the current git branch. The function below puts it at the very beginning. Run the cell: what happens to the cache?

In [ ]:
def build_messages_naive(question: str, developer: str, branch: str) -> list[dict]:
    """Builds the request with session information at the beginning of the system prompt."""
    session_info = f"Current time: {datetime.now().isoformat()}\nDeveloper: {developer}\nGit branch: {branch}\n\n"
    return [
        {"role": "system", "content": session_info + SYSTEM_PROMPT_WITH_CODE},
        {"role": "user", "content": question},
    ]


for question in ["Which function checks the business hours?", "Where is the API key of a request checked?"]:
    response = ask(build_messages_naive(question, developer="alice", branch="fix/WORK-412-naive-datetimes"))
    print_usage(question[:28], response)

The time changes with every request – and because it comes first, the prefix is different every time. Not a single token is cached.

Your task is to build the request so that the same information reaches the model, but the cache works again.

>**Note:** If your system prompt is the same as in the first cell, it may already be in the cache. Otherwise, the first request fills the cache. Caching is best effort: if you see a miss, run the cell again.

In [ ]:
# TODO: Build the request so that the stable part comes first and the dynamic session information last
def build_messages(question: str, developer: str, branch: str) -> list[dict]:
    """Builds the request with a stable system prompt; session information goes into the user message."""
    session_info = f"Current time: {datetime.now().isoformat()}\nDeveloper: {developer}\nGit branch: {branch}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT_WITH_CODE},
        {"role": "user", "content": f"{session_info}\n\n{question}"},
    ]


for question in ["Which function checks the business hours?", "Where is the API key of a request checked?"]:
    response = ask(build_messages(question, developer="alice", branch="fix/WORK-412-naive-datetimes"))
    print_usage(question[:28], response)

What does this mean for a team? Let's extrapolate the costs with the token numbers of the last response – as in the augmentation lesson: 1,000 requests per developer and day, a team of 10 developers.

In [ ]:
model_info = litellm.get_model_info(generation_model)
prompt_tokens = response.usage.prompt_tokens
from_cache = cached_tokens(response)
requests_per_day = 1_000 * 10

costs_without_cache = prompt_tokens * model_info["input_cost_per_token"]
costs_with_cache = (prompt_tokens - from_cache) * model_info["input_cost_per_token"] + from_cache * model_info["cache_read_input_token_cost"]

print(f"Input costs per request without cache: {costs_without_cache:.6f} USD")
print(f"Input costs per request with cache:    {costs_with_cache:.6f} USD")
print(f"Per day for the team: {costs_without_cache * requests_per_day:.2f} USD vs. {costs_with_cache * requests_per_day:.2f} USD")

What did we learn?
* Order the context from **stable to dynamic**: system prompt, tool declarations, instructions and code first; session information, conversation and the current question last.
* Things that break the cache: timestamps or ids at the beginning, tool declarations in a different order, editing earlier messages of the conversation. Appending to the conversation keeps the cache – that's why the tool loop is cache friendly.
* Caching makes the context cheaper, but not smaller: the model still has to read all 12,000 tokens – and its attention is spread over all of them.

### Exercise 02: Progressive disclosure with skills

Teams have many recurring tasks with their own conventions: commit messages, merge requests, regression tests, security reviews. A coding assistant should follow them – but putting all instructions into every request would make the stable part big and noisy.

**Skills** solve this with *progressive disclosure*: the assistant first sees only a small index and loads the details when it needs them. A skill is a folder with a `SKILL.md` file in the open [Agent Skills](https://agentskills.io) format – a short header with `name` and `description`, followed by the instructions in Markdown:

```
data/skills/
├── write-commit-message/
│   └── SKILL.md
├── write-merge-request-description/
│   ├── SKILL.md
│   └── assets/template.md
└── ...
```

| Level | What | When is it in the context? |
|---|---|---|
| 1 – index | `name` and `description` of every skill | always (system prompt) |
| 2 – instructions | the rest of `SKILL.md` | when the model calls the tool `load_skill` |
| 3 – resources | additional files of a skill, e.g. templates | when the instructions refer to them |

This is the same idea as the knowledge base search in the function calling lesson: the model decides itself *when* it needs knowledge. Let's look at the skills of Team Workplace.

In [ ]:
skills = load_skills(SKILLS_PATH)

for skill in skills.values():
    print(f"{skill['name']:34} {count_tokens(skill['instructions']):5} tokens")
print()
print((SKILLS_PATH / "write-commit-message" / "SKILL.md").read_text())

Your task is to complete the function that builds the system prompt:

1. Build the **index** (level 1): one line per skill with its name and description.
2. Explain in the system prompt **how the model should use the skills** – in your own words. The model has a tool `load_skill` to load the instructions of a skill.

In [ ]:
# TODO: Build the skill index and explain the model how to use the skills
def build_system_prompt(skills: dict) -> str:
    skill_index = ""
    for skill in skills.values():
        skill_index += f"- {skill['name']}: {skill['description']}\n"

    return f"""You are a coding assistant for Team Workplace, the team that owns Roomie, the meeting room booking service.

## Skills
Skills are instructions for recurring tasks of the team. Before you start a task, check whether one of the following skills matches it.
If one does, call the tool load_skill first and follow its instructions exactly. Don't load skills you don't need.

{skill_index}"""


SKILLS_SYSTEM_PROMPT = build_system_prompt(skills)
print(SKILLS_SYSTEM_PROMPT)

all_instructions = "\n\n".join(skill["instructions"] for skill in skills.values())
print(f"System prompt with the index: about {count_tokens(SKILLS_SYSTEM_PROMPT)} tokens")
print(f"All instructions of all skills: about {count_tokens(all_instructions)} tokens")

Level 2 is a normal tool, just like `search_knowledge_base` in the function calling lesson. The function returns the instructions of a skill; for an unknown name, it returns an error message, so that the model can correct itself.

In [ ]:
def load_skill(name: str) -> str:
    """Returns the instructions of a skill."""
    if name not in skills:
        return f"Error: unknown skill '{name}'. Available skills: {list(skills)}"
    return skills[name]["instructions"]


load_skill_tool = {
    "type": "function",
    "function": {
        "name": "load_skill",
        "description": "Loads the full instructions of a skill. Call it before you work on a task that matches the description of a skill.",
        "parameters": {
            "type": "object",
            "properties": {"name": {"type": "string", "description": "The name of the skill, as listed in the system prompt."}},
            "required": ["name"],
        },
    },
}

Let's test it. The developer from our session has fixed `WORK-412` and asks for a commit message. The skill `write-commit-message` contains a convention that the model can't know without it: the `Assisted-by:` line at the end. `run_assistant` is the tool loop from the function calling lesson – it prints every tool call.

In [ ]:
WORK_412_DIFF = '''diff --git a/roomie/errors.py b/roomie/errors.py
@@ -70,6 +70,11 @@ class SlotAlignmentError(RoomieError):
+class NaiveTimestampError(RoomieError):
+    status_code = 422
+    code = "ROOM-422-NAIVE-TIME"
+
+
 class DurationError(RoomieError):
diff --git a/roomie/services/booking_rules.py b/roomie/services/booking_rules.py
@@ -21,6 +22,15 @@ from roomie.models import Booking, BookingCreate, Room
+def check_timezone_aware(start: datetime, end: datetime) -> None:
+    """Start and end must have a timezone offset (API-07, ADR-0004)."""
+    for moment in (start, end):
+        if moment.tzinfo is None or moment.utcoffset() is None:
+            raise NaiveTimestampError(
+                "Timestamps must include a timezone offset, e.g. 2026-03-03T10:00:00+01:00."
+            )
+
+
@@ -106,6 +116,7 @@ def validate_new_booking(
     """Runs all booking rules in order. Raises the first violation."""
+    check_timezone_aware(request.start, request.end)
     check_slot_alignment(request.start, request.end)
'''

answer = run_assistant(
    SKILLS_SYSTEM_PROMPT,
    f"I fixed WORK-412. Please write the commit message for this diff:\n```diff\n{WORK_412_DIFF}```",
    tools={"load_skill": load_skill},
    tool_declarations=[load_skill_tool],
    model=generation_model,
)
print(f"\n{answer}\n")
print(f"{'✅' if 'Refs: WORK-412' in answer else '❌'} Ticket reference 'Refs: WORK-412'")
print(f"{'✅' if 'Assisted-by:' in answer else '❌'} 'Assisted-by:' line (only in the skill)")

### Exercise 03: Write your own skill

Writing skills is something you will do in your daily work – no Python needed, only Markdown. According to `docs/error_codes.md` of Roomie, every new error code must be announced in the Slack channel `#team-workplace`, because the clients (web app, Slack bot, door displays) need a message for it. Your task is to write a skill `announce-error-code` for this announcement.

* `name` – must be the name of the folder: `announce-error-code`
* `description` – *what* the skill does and *when* to use it. This is the only thing the model sees before it loads the skill – it decides whether your skill is used at all.
* Instructions – define a **format** for the announcement, with something recognizable (e.g. a fixed first line), so that you can see whether the model followed your skill.

Your skill is saved in a temporary folder next to the skills of the team.

>**How `%%writefile` works:** If the first line of a code cell is `%%writefile <path>`, Jupyter doesn't run the cell as Python – it saves the rest of the cell as a file under this path. `{MY_SKILL_FILE}` in curly braces is replaced by the value of the Python variable. Edit the text in the cell and run it again to overwrite the file.

In [ ]:
MY_SKILLS_PATH = Path(tempfile.gettempdir()) / "my_skills"
MY_SKILL_FILE = MY_SKILLS_PATH / "announce-error-code" / "SKILL.md"
MY_SKILL_FILE.parent.mkdir(parents=True, exist_ok=True)
print(MY_SKILL_FILE)

In [ ]:
%%writefile {MY_SKILL_FILE}
---
name: announce-error-code
description: Writes the Slack announcement of a new error code of Roomie for the team channel #team-workplace. Use it when the developer added an error code and wants to announce it.
---

# Announce a new error code

Every new error code of Roomie must be announced in `#team-workplace`, because the clients
(web app, Slack bot, door displays, mobile app) show their own message for every code.

Write the announcement in exactly this format:

📢 New error code in Roomie: `<code>`
*HTTP status:* <status>
*When:* <one sentence from the perspective of a client developer>
*What clients have to do:* <e.g. add a localized message for the code>
*Ticket:* <ticket id>

Rules:
- No more than these five lines, no code blocks, no greeting.
- Never mention names of users or meeting titles (SEC-04).

Now load the skills again – the team's and yours – and rebuild the system prompt. Then ask the assistant. Did it load your skill? Did it follow your format?

>**Tip:** If your skill is not loaded, improve the `description` – not the question.

In [ ]:
skills = load_skills(SKILLS_PATH) | load_skills(MY_SKILLS_PATH)
SKILLS_SYSTEM_PROMPT = build_system_prompt(skills)

answer = run_assistant(
    SKILLS_SYSTEM_PROMPT,
    "I added the error code ROOM-422-NAIVE-TIME for WORK-412: the API answers with 422 if start or end of a booking "
    "has no timezone offset. Please write the announcement for our team channel.",
    tools={"load_skill": load_skill},
    tool_declarations=[load_skill_tool],
    model=generation_model,
)
print(f"\n{answer}\n")

# TODO: Enter a text that your format requires
MY_FORMAT_MARKER = "📢 New error code in Roomie"
print(f"{'✅' if MY_FORMAT_MARKER in answer else '❌'} Follows the format of my skill ({MY_FORMAT_MARKER!r})")

What did we learn?
* The skill index costs a few hundred tokens instead of a few thousand – and it is the same in every request, so it is cached, too. The instructions arrive as a tool result *at the end* of the context, only when they are needed.
* The `description` decides whether a skill is used – just like the description of a tool. Write it like a prompt: *what* the skill does and *when* to use it.
* Level 3 works the same way: the skill `write-merge-request-description` has a template in `assets/`, and a tool like `read_skill_file(skill, path)` can load it when the instructions say so. Careful: the path comes from the model – such a tool must only read files *inside* the skill folder (remember the guardrails lesson).
* Skills are just files. The same folders work in other harnesses that support the Agent Skills format (e.g. Claude Code, GitHub Copilot) – like an MCP server, one skill can serve many assistants.
* Everything the model reads – also skills from a shared repository – is part of the context and can contain injected instructions. Review skills like code.

### Exercise 04: History condensing

Back to our session. Every request sends the complete conversation – including the server log, three files and the test output. At some point the session reaches the token budget of the harness (or the context window of the model). Then the harness has to shrink the history. There are three common strategies:

| Strategy | How | Costs | Risk |
|---|---|---|---|
| Sliding window | drop the oldest turns | free | important information from the beginning is lost |
| Clear tool results | replace old tool results with a placeholder | free | the model has to call the tool again if it needs the result |
| Summarize | let the model write a summary of the old turns | one request | details that the summary leaves out are lost |

To compare them, the developer asks a follow-up question that needs the **decision** from the beginning of the session. The assistant may use `read_file` to answer it.

In [ ]:
FOLLOW_UP = "Now write the new exception class for roomie/errors.py and the new row for docs/error_codes.md for the error we agreed on."


def ask_follow_up(label: str, messages: list[dict]) -> None:
    """Sends the follow-up question with the given history and checks the answer for the agreed error code."""
    result = run_tool_loop(
        messages + [{"role": "user", "content": FOLLOW_UP}],
        tools={"read_file": read_file},
        tool_declarations=[READ_FILE_TOOL],
        model=generation_model,
    )
    print(f"\n{result.answer}\n")
    print(f"{label}: {result.requests} request(s), {result.prompt_tokens} prompt tokens, {result.costs:.6f} USD")
    print(f"{'✅' if 'ROOM-422-NAIVE-TIME' in result.answer else '❌'} Uses the agreed error code ROOM-422-NAIVE-TIME")


print(f"Full history: about {history_tokens(session)} tokens")

#### Example: Sliding window

The simplest strategy keeps the system prompt and the last turns. A turn starts with a message of the developer – we never cut between a tool call and its result, the provider would reject the request.

In [ ]:
first_kept = turn_starts(session)[-3]  # the message where the third last turn starts
window = [session[0]] + session[first_kept:]

print(f"Sliding window: about {history_tokens(window)} tokens")
ask_follow_up("Sliding window", window)

The history is small – but the decision is gone. The model may read `roomie/errors.py` again, but the agreed code isn't in any file: it invents an error code that looks plausible, but nobody agreed on it.

#### Example: Clear old tool results

Tool results are the biggest part of the history, and they can be fetched again: the file is still there. `clear_old_tool_results` replaces all but the last tool results with a short placeholder:

In [ ]:
cleared = clear_old_tool_results(session, keep_last=1)
print(f"Full history:        about {history_tokens(session)} tokens")
print(f"Without old results: about {history_tokens(cleared)} tokens")
print(cleared[9]["content"])

That helps, and it is free – but the pasted server log and the test output are not tool results and stay in the history. For that we need a summary.

#### Summarize (condensing)

`condense_history` replaces the old turns by a summary written by the model and keeps the last turns word by word. The summary goes to the end of the system prompt. `format_transcript` turns the old messages into plain text and shortens long contents (files, logs).

Whether a summary is useful depends on its **prompt**. Your task is to write it:

1. Start with `"Summarize the conversation."` and look at the result.
2. Improve the prompt: Which sections should a summary have, so that the conversation can continue? What must be kept *exactly*? What can be left out, because it can be fetched again?

In [ ]:
# TODO: Write the prompt for the summary
CONDENSE_PROMPT = """You condense the history of a conversation between a developer and a coding assistant, so that the conversation can continue with a much shorter context.

Write a summary with these sections:
## Goal
## Decisions (with the exact names, codes and values, and the reason)
## Relevant files and functions
## Done so far
## Open tasks

Rules:
- Keep exact identifiers: file, function and class names, error codes, ticket ids.
- Do not copy file contents, logs or test output - say where they can be found again instead.
- At most 250 words."""


def condense_history(messages: list[dict], keep_last_turns: int = 2) -> list[dict]:
    """Replaces all but the last turns with a summary at the end of the system prompt."""
    first_kept = turn_starts(messages)[-keep_last_turns]
    old_messages, recent_messages = messages[1:first_kept], messages[first_kept:]
    response = call_with_retry(
        completion,
        model=generation_model,
        messages=[{"role": "system", "content": CONDENSE_PROMPT}, {"role": "user", "content": format_transcript(old_messages)}],
        temperature=0.0,
    )
    summary = response.choices[0].message.content
    system_prompt = f"{messages[0]['content']}\n\n## Summary of the conversation so far\n{summary}"
    return [{"role": "system", "content": system_prompt}] + recent_messages


condensed = condense_history(session)
print(condensed[0]["content"])
print(f"\nCondensed history: about {history_tokens(condensed)} tokens")
for identifier in ["WORK-412", "ROOM-422-NAIVE-TIME", "NaiveTimestampError", "check_timezone_aware", "validate_new_booking"]:
    print(f"{'✅' if identifier in condensed[0]['content'] else '❌'} {identifier}")

Does the assistant still know the decision?

In [ ]:
ask_follow_up("Condensed history", condensed)

For comparison: the full history (the most expensive variant).

In [ ]:
ask_follow_up("Full history", session)

What did we learn?
* Condensing keeps what matters for the task and drops what can be fetched again. In our short session even a simple prompt keeps the decision – in long sessions with many decisions, a structured prompt that names what must survive (decisions, exact identifiers) makes the summaries reliable.
* Clearing old tool results is free and safe for results that the model can fetch again. It pairs well with tools: *just in time* instead of *just in case*.
* Condensing rewrites the beginning of the conversation – the **prompt cache** of exercise 01 is gone afterwards. That's why harnesses condense rarely and in big steps (e.g. when 80% of the budget is used) instead of after every turn. Claude Code, for example, compacts the conversation automatically near the limit or on demand with `/compact`.
* A summary always loses details. If a fact is important, it should not depend on the summary: write it down – in the ticket, the code, a file. That's what agents do with *notes*, which we will see in the next lesson.

### Learnings
* The context window is the working memory of the model – and the harness decides what is in it. **Context engineering** is the design of this decision: what goes in, when, in which order, and what stays out.
* Every request resends the complete context. The costs of a session grow faster than its length, and long, noisy contexts make the answers worse (*context rot*).
* Order the context from **stable to dynamic** to benefit from **prompt caching**: system prompt, tools and instructions first, session information and conversation last.
* Use **progressive disclosure**: give the model an index (skills, tools, files) and let it load the details just in time. Writing good skills – especially their descriptions – is a skill in itself.
* **Condense** long histories: clear tool results that can be fetched again, summarize the rest with a prompt that keeps decisions and exact identifiers.
* The same principles apply to tools: dozens of MCP tools with long descriptions fill the context, too. Some harnesses therefore select the tools per request or load them on demand.
* Further reading: [Thoughtworks Technology Radar – Context engineering](https://www.thoughtworks.com/radar/techniques/context-engineering)
* In the next lesson, the assistant becomes an **agent** that works on a task for many steps on its own – and long-running agents need all of these techniques, plus some more: notes as external memory and sub-agents that work in a separate context.